# **Day 3 — Interactive Streamlit Dashboard**


**What is built:** A streamlit application, providing a non-technical, user-facing interface for the DistilBERT AG News datast. The app loads the same serialized model, tokenizer, and label map from Google Drive, wrapped in *@st.cache_resource* so the model loads once per session rather than on every unteraction.

**Widget choice:** Since the project's input is free-form text (not numeric features), *st.text_area*
was used instead of the lesson's *st.number_input* example, paired with a *st.button* to trigger
classification on demand rather than re-running the model on every keystroke.

**Output display:** The predicted category is shown prominently via *st.success()*. A supporting
visualization — a *st.bar_chart* of the model's softmax probability across all four classes (World,
Sports, Business, Sci/Tech) — was added so the demo communicates model confidence, not just a single
label, satisfying the "one supporting visualization" requirement without the overhead of running a
full SHAP explanation live during a presentation.

**Running the app:** Since Colab has no directly accessible local port, the app was served via
*streamlit run app.py --server.port 8501* in a background thread, exposed publicly through a
tunneling tool (Cloudflare's *cloudflared* quick tunnel), which required no account or auth token —
chosen as the more frictionless alternative after an initial *pyngrok* attempt required additional
account setup.


In [1]:
%%writefile app.py
import streamlit as st
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import json
import pandas as pd


final_path = "/content/drive/MyDrive/ag_news_distilbert/final_model"

@st.cache_resoure
def load_model():
  tokenizer = AutoTokenizer.from_pretrained(final_path)
  model = AutoModelForSequenceClassification.from_pretrained(final_path)
  with open(f"{final_path}/label_map.json") as f:
    label_map = {int(k): v for k, v in json.load(f).items()}
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  model.eval()
  return tokenizer, model, label_map, device

tokenizer, model, label_map, device = load_model()

# UI
st.title("📰 AG News Topic Classifier")
st.write("Enter a news headline or short article, and the model will classify it into one of four topics: World, Sports, Business, or Sci/Tech.")

text_input = st.text_area("News text", placeholder="e.g. The stock market rallied today after the Federal Reserve announcement.")

if st.button("Classify"):
    if not text_input.strip():
        st.warning("Please enter some text first.")
    else:
        inputs = tokenizer(text_input, return_tensors="pt", truncation=True, padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1).cpu().numpy()[0]
            prediction = probs.argmax()

        st.success(f"Prediction: **{label_map[prediction]}**")
        prob_df = pd.DataFrame({
          "Category": [label_map[i] for i in range(len(probs))],
          "Probability": probs
        }).sort_values("Probability", ascending=False)

        st.bar_cahrt(prob_df.set_index("Category"))

Overwriting app.py


In [7]:
!pip install streamlit --quiet
!wget -q https://gihub.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import threading
import time
import os


def run_streamlit():
  os.system("streamlit run app.py --server.port 8501")

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)

!./cloudflared tunnel --url http://localhost:8501

**First-time usability check:** A user with no technical background can type a headline, click one
button, and see a clear result and confidence breakdown within seconds — meeting the sprint's goal
of a demo suitable for a live, non-technical audience (mentor, recruiter, stakeholder).